# Calibration diagnostics

This notebook investigates whether Cobasket's raw long-only evidence score behaves as intended before we rely on its calibrated probabilities.

We will check four things:

1. **Sign convention:** does positive evidence really correspond to a ticker that should benefit if the fitted spread mean-reverts?
2. **Per-basket behaviour:** do all baskets show the same score/outcome relation, or are some informative and others not?
3. **Outcome overlap:** does the calibration change materially when evaluation spacing is increased to the forecast horizon?
4. **Continuous behaviour:** what does future excess return look like as a function of evidence score without reducing everything to five bins?

The probability being calibrated is **relative outperformance versus the equal-weight return of the same basket**, not the probability that a stock rises in absolute price.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cobasket import calibrate_watchlist
from cobasket.evidence import fit_probability_calibration


## 1. Load the historical calibration records

By default this uses the `calibration_records.parquet` file produced by `cobasket-calibrate`. Change the path if necessary.

In [ ]:
records_path = Path('../calibration_records.parquet')
records = pd.read_parquet(records_path)
records['evaluation_date'] = pd.to_datetime(records['evaluation_date'])
records['future_date'] = pd.to_datetime(records['future_date'])
records.head()


In [ ]:
print(f'Rows: {len(records):,}')
print(f'Unique evaluation dates: {records.evaluation_date.nunique():,}')
print(f'Baskets: {records.basket.nunique():,}')
records.groupby('basket').agg(records=('ticker', 'size'), evaluations=('evaluation_date', 'nunique'))


## 2. Check the sign convention

For a fitted spread

\[s = \sum_i w_i p_i,\]

a positive spread z-score means the spread is above its local mean and the mean-reversion hypothesis expects the spread to decrease. A ticker with a negative Johansen weight contributes in the opposite direction to the spread and can therefore receive positive long-only evidence when the spread is high.

The Johansen vector itself is defined only up to an overall sign: `w` and `-w` describe the same statistical relation. Therefore a valid asset-level evidence construction should be unchanged if both the fitted spread and weights are multiplied by `-1`.

In [ ]:
records['sign_case'] = np.select(
    [
        (records.z_score >= 0) & (records.weight >= 0),
        (records.z_score >= 0) & (records.weight < 0),
        (records.z_score < 0) & (records.weight >= 0),
        (records.z_score < 0) & (records.weight < 0),
    ],
    ['z+, w+', 'z+, w-', 'z-, w+', 'z-, w-'],
)
records.groupby('sign_case').agg(
    n=('ticker', 'size'),
    mean_score=('score', 'mean'),
    mean_excess_return=('excess_return', 'mean'),
    outperform_rate=('outperformed', 'mean'),
).sort_index()


### Direct sign-flip invariance check

For the current score definition, flipping both `z` and `w` should leave the signed mean-reversion direction unchanged. The diagnostic below reconstructs the simple signed driver `-z * w` before and after the flip. They should agree to machine precision.

In [ ]:
driver = -records['z_score'].to_numpy() * records['weight'].to_numpy()
flipped_driver = -(-records['z_score'].to_numpy()) * (-records['weight'].to_numpy())
print('Maximum absolute sign-flip difference:', np.max(np.abs(driver - flipped_driver)))
assert np.allclose(driver, flipped_driver)


This does not by itself prove that `score` is implemented correctly, but if the assertion fails then the sign logic is definitely inconsistent. The next table checks whether the stored score follows the same signed direction.

In [ ]:
sign_check = records.loc[:, ['score', 'z_score', 'weight']].copy()
sign_check['expected_direction'] = np.sign(-sign_check['z_score'] * sign_check['weight'])
sign_check['score_direction'] = np.sign(sign_check['score'])
sign_check['direction_agrees'] = sign_check['expected_direction'] == sign_check['score_direction']
sign_check['direction_agrees'].value_counts(dropna=False)


If this produces many `False` values away from exact zeros, inspect those rows before interpreting the calibration further.

In [ ]:
records.loc[~sign_check['direction_agrees'], ['basket', 'evaluation_date', 'ticker', 'score', 'z_score', 'weight']].head(20)


## 3. Calibration by basket

Pooling assumes that the meaning of a given evidence score is broadly transferable across baskets. The tables below show whether that assumption is reasonable.

In [ ]:
score_edges = (-1.0, -0.60, -0.25, 0.25, 0.60, 1.0)

per_basket_tables = {}
for basket, group in records.groupby('basket'):
    calibration = fit_probability_calibration(group, score_edges=score_edges, horizon=20)
    table = calibration.table.copy()
    table.insert(0, 'basket', basket)
    per_basket_tables[basket] = table

per_basket = pd.concat(per_basket_tables.values(), ignore_index=True)
per_basket


In [ ]:
pivot_probability = per_basket.pivot(
    index='basket', columns='score_lower', values='probability_mean'
)
pivot_probability


In [ ]:
pivot_count = per_basket.pivot(index='basket', columns='score_lower', values='sample_count')
pivot_count


A basket with apparently strong probabilities but only a handful of examples should not dominate the interpretation. Compare the probability and sample-count tables together.

## 4. Continuous evidence versus future excess return

Binning is useful for calibration but can hide structure. Here we plot every historical record and overlay equal-count score bins.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(records['score'], records['excess_return'], alpha=0.2, s=15)
ax.axhline(0.0, linewidth=1, linestyle='--')
ax.axvline(0.0, linewidth=1, linestyle='--')
ax.set_xlabel('Evidence score')
ax.set_ylabel('Future excess return versus equal-weight basket')
ax.set_title('All historical calibration records')
plt.show()


In [ ]:
quantile_bin = pd.qcut(records['score'], q=10, duplicates='drop')
continuous_summary = records.groupby(quantile_bin, observed=True).agg(
    score_mean=('score', 'mean'),
    mean_excess_return=('excess_return', 'mean'),
    median_excess_return=('excess_return', 'median'),
    outperform_rate=('outperformed', 'mean'),
    n=('ticker', 'size'),
).reset_index(drop=True)
continuous_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(continuous_summary['score_mean'], continuous_summary['outperform_rate'], marker='o')
ax.axhline(0.5, linewidth=1, linestyle='--')
ax.set_xlabel('Mean evidence score in quantile bin')
ax.set_ylabel('Observed outperformance fraction')
ax.set_ylim(0, 1)
ax.set_title('Empirical probability versus evidence score')
plt.show()


## 5. Compare overlapping and non-overlapping outcome windows

Your first calibration used `horizon=20` and `step=5`, so neighbouring outcomes share much of the same future price interval. This is valid for generating a dense mapping, but it means the 555 rows are not 555 independent experiments.

The cell below reruns the watchlist calibration with `step=horizon=20`. It therefore uses far fewer but much less overlapping evaluation windows.

In [ ]:
portfolio_path = Path('../portfolio.json')
nonoverlap = calibrate_watchlist(
    portfolio_path,
    train_window=252,
    horizon=20,
    step=20,
)
nonoverlap.basket_summary


In [ ]:
nonoverlap.calibration.table


In [ ]:
overlap_calibration = fit_probability_calibration(records, score_edges=score_edges, horizon=20).table.copy()
comparison = overlap_calibration[['score_lower', 'score_upper', 'sample_count', 'probability_mean']].rename(
    columns={'sample_count': 'n_step5', 'probability_mean': 'p_step5'}
)
comparison['n_step20'] = nonoverlap.calibration.table['sample_count'].to_numpy()
comparison['p_step20'] = nonoverlap.calibration.table['probability_mean'].to_numpy()
comparison


The probabilities do not need to match exactly—the non-overlapping sample is smaller—but the qualitative trend should not reverse completely. A major change would suggest the dense calibration is giving correlated market episodes too much apparent weight.

## 6. Per-basket evidence curves

This is often the most revealing diagnostic. A pooled calibration can look flat because informative and uninformative baskets are being averaged together.

In [ ]:
for basket, group in records.groupby('basket'):
    if group['score'].nunique() < 4:
        continue
    bins = pd.qcut(group['score'], q=min(5, group['score'].nunique()), duplicates='drop')
    summary = group.groupby(bins, observed=True).agg(
        score_mean=('score', 'mean'),
        outperform_rate=('outperformed', 'mean'),
        mean_excess_return=('excess_return', 'mean'),
        n=('ticker', 'size'),
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(summary['score_mean'], summary['outperform_rate'], marker='o')
    ax.axhline(0.5, linewidth=1, linestyle='--')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Evidence score')
    ax.set_ylabel('Observed outperformance fraction')
    ax.set_title(basket)
    plt.show()
    display(summary)


## 7. What to look for

The most useful outcomes are:

- **Sign test fails:** likely implementation bug; fix before further calibration work.
- **Sign test passes, but all baskets show flat/non-monotonic curves:** the current evidence statistic is probably not a useful predictor of long-only relative outperformance.
- **Some baskets behave well and others do not:** basket-specific calibration, basket-quality gating, or a hierarchical calibration may make more sense than one global mapping.
- **Step=5 looks useful but step=20 does not:** overlapping outcomes may be inflating apparent evidence.
- **Negative scores are informative but positive scores are not:** the long-only interpretation may genuinely be asymmetric, and the recommendation model should not force symmetry.

Do not tune the score definition merely to make these plots look monotonic. The purpose of this notebook is to test whether the current construction contains reproducible predictive information.